In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install einops umap-learn scipy scikit-learn matplotlib tqdm

In [3]:
from pathlib import Path
import os
import sys
import shutil
import zipfile
import json
import subprocess
import time
import torch

# ------------------------------------------------------------------
# Drive workspace
# ------------------------------------------------------------------
XAI_ROOT = Path('/content/drive/MyDrive/colab/lsa_convtok_vit/xai')
DRIVE_SCRIPTS_ROOT = XAI_ROOT / 'scripts'
DRIVE_PREPROCESSED_ROOT = XAI_ROOT / 'preprocessed'
DRIVE_OUTPUT_ROOT = XAI_ROOT / 'output'

EXPERIMENT = 'lsa_convtok'
DRIVE_EXPERIMENT_DIR = DRIVE_OUTPUT_ROOT / EXPERIMENT
DRIVE_MODEL_PATH = DRIVE_EXPERIMENT_DIR / 'final_model_lsa_convtok.pth'
ZIP_PATH = DRIVE_PREPROCESSED_ROOT / 'preprocessed_384.zip'

# ------------------------------------------------------------------
# Local Colab workspace. These paths avoid Drive read/write instability.
# ------------------------------------------------------------------
LOCAL_WORK_ROOT = Path('/content/lsa_convtok_xai_run')
LOCAL_SCRIPTS_DIR = LOCAL_WORK_ROOT / 'scripts'
LOCAL_DATA_ROOT = LOCAL_WORK_ROOT / 'data'
LOCAL_ZIP_PATH = LOCAL_WORK_ROOT / 'preprocessed_384.zip'
LOCAL_OUTPUT_ROOT = LOCAL_WORK_ROOT / 'output'
LOCAL_EXPERIMENT_DIR = LOCAL_OUTPUT_ROOT / EXPERIMENT
LOCAL_MODEL_PATH = LOCAL_EXPERIMENT_DIR / 'final_model_lsa_convtok.pth'
LOCAL_ANALYSIS_DIR = LOCAL_EXPERIMENT_DIR / 'analysis'

print('XAI_ROOT:', XAI_ROOT)
print('DRIVE_SCRIPTS_ROOT:', DRIVE_SCRIPTS_ROOT)
print('DRIVE_PREPROCESSED_ROOT:', DRIVE_PREPROCESSED_ROOT)
print('DRIVE_OUTPUT_ROOT:', DRIVE_OUTPUT_ROOT)
print('DRIVE_MODEL_PATH:', DRIVE_MODEL_PATH)
print('ZIP_PATH:', ZIP_PATH)
print('\nLOCAL_WORK_ROOT:', LOCAL_WORK_ROOT)
print('LOCAL_OUTPUT_ROOT:', LOCAL_OUTPUT_ROOT)
print('LOCAL_MODEL_PATH:', LOCAL_MODEL_PATH)

assert XAI_ROOT.exists(), f'XAI root not found: {XAI_ROOT}'
assert DRIVE_SCRIPTS_ROOT.exists(), f'Scripts folder not found: {DRIVE_SCRIPTS_ROOT}'
assert DRIVE_PREPROCESSED_ROOT.exists(), f'Preprocessed folder not found: {DRIVE_PREPROCESSED_ROOT}'
assert DRIVE_MODEL_PATH.exists(), f'Model artifact not found: {DRIVE_MODEL_PATH}'
assert ZIP_PATH.exists(), f'Dataset ZIP not found: {ZIP_PATH}'

LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print('\nPath checks passed.')


XAI_ROOT: /content/drive/MyDrive/colab/lsa_convtok_vit/xai
DRIVE_SCRIPTS_ROOT: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/scripts
DRIVE_PREPROCESSED_ROOT: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/preprocessed
DRIVE_OUTPUT_ROOT: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/output
DRIVE_MODEL_PATH: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/output/lsa_convtok/final_model_lsa_convtok.pth
ZIP_PATH: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/preprocessed/preprocessed_384.zip

LOCAL_WORK_ROOT: /content/lsa_convtok_xai_run
LOCAL_OUTPUT_ROOT: /content/lsa_convtok_xai_run/output
LOCAL_MODEL_PATH: /content/lsa_convtok_xai_run/output/lsa_convtok/final_model_lsa_convtok.pth

Path checks passed.


In [4]:

required_scripts = [
    'analysis_attention.py',
    'extractor.py',
    'visualization.py',
    'config.py',
    'embeddings.py',
    'helper.py',
    'stratified_sampling.py',
    'training_utils.py',
    'lsa_convtok_vit.py',
    'lsa_attention.py',
]

missing = [f for f in required_scripts if not (DRIVE_SCRIPTS_ROOT / f).exists()]
if missing:
    raise FileNotFoundError('Missing required scripts: ' + ', '.join(missing))

if LOCAL_SCRIPTS_DIR.exists():
    shutil.rmtree(LOCAL_SCRIPTS_DIR)
shutil.copytree(DRIVE_SCRIPTS_ROOT, LOCAL_SCRIPTS_DIR)

if str(LOCAL_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_SCRIPTS_DIR))

print('Copied scripts to:', LOCAL_SCRIPTS_DIR)
print('Scripts copied:')
for f in required_scripts:
    print(' -', f)

# Import check
import config
print('\nLocal config import OK.')


Copied scripts to: /content/lsa_convtok_xai_run/scripts
Scripts copied:
 - analysis_attention.py
 - extractor.py
 - visualization.py
 - config.py
 - embeddings.py
 - helper.py
 - stratified_sampling.py
 - training_utils.py
 - lsa_convtok_vit.py
 - lsa_attention.py

Local config import OK.


In [5]:

def copy_file_with_retries(src: Path, dst: Path, max_attempts: int = 5, sleep_s: int = 5) -> None:
    last_error = None
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_attempts + 1):
        try:
            if dst.exists():
                dst.unlink()
            print(f'Copying {src.name} to local disk, attempt {attempt}/{max_attempts}...')
            shutil.copy2(src, dst)
            src_size = src.stat().st_size
            dst_size = dst.stat().st_size
            print(f'  source size: {src_size / 1024 / 1024:.2f} MB')
            print(f'  local size:  {dst_size / 1024 / 1024:.2f} MB')
            if src_size != dst_size:
                raise IOError(f'Size mismatch: source={src_size}, local={dst_size}')
            return
        except Exception as e:
            last_error = e
            print(f'  attempt {attempt} failed: {repr(e)}')
            if dst.exists():
                try:
                    dst.unlink()
                except Exception:
                    pass
            time.sleep(sleep_s)
    raise RuntimeError(f'Failed to copy {src} after {max_attempts} attempts') from last_error

copy_file_with_retries(DRIVE_MODEL_PATH, LOCAL_MODEL_PATH)

# Validate from local disk, not Drive.
ckpt = torch.load(LOCAL_MODEL_PATH, map_location='cpu', weights_only=False)
print('\nTop-level model artifact keys:')
print(sorted(ckpt.keys()))

required_model_keys = ['model_state_dict', 'mean', 'std', 'config', 'metadata', 'training_history']
missing_model_keys = [k for k in required_model_keys if k not in ckpt]
assert not missing_model_keys, f'Model artifact missing keys: {missing_model_keys}'

state = ckpt['model_state_dict']
assert any('tokenizer' in k for k in state.keys()), 'Missing ConvTokenizer weights.'
assert any('temperature' in k for k in state.keys()), 'Missing LSA temperature parameters.'
assert 'linear_head.weight' in state, 'Missing linear_head.weight.'

print('\nModel artifact validation passed.')
print('model_variant:', ckpt.get('model_variant'))
print('optimal_epochs:', ckpt.get('optimal_epochs'))
print('mean:', ckpt.get('mean'))
print('std:', ckpt.get('std'))
print('linear_head.weight shape:', tuple(state['linear_head.weight'].shape))

metadata = ckpt.get('metadata', {})
arch_meta = metadata.get('arch_metadata', {}) if isinstance(metadata, dict) else {}
if arch_meta:
    print('\nArchitecture metadata:')
    for k in ['experiment_name', 'model_class', 'attention_type', 'arch_hash', 'image_size', 'channels']:
        print(f'  {k}:', arch_meta.get(k))


Copying final_model_lsa_convtok.pth to local disk, attempt 1/5...
  source size: 20.69 MB
  local size:  20.69 MB

Top-level model artifact keys:
['architecture', 'class_to_idx', 'config', 'mean', 'metadata', 'model_state_dict', 'model_variant', 'optimal_epochs', 'std', 'training_history']

Model artifact validation passed.
model_variant: SimpleViT_LSA_ConvTok
optimal_epochs: 43
mean: [0.23691008985042572]
std: [0.007409718818962574]
linear_head.weight shape: (2, 256)

Architecture metadata:
  experiment_name: lsa_convtok
  model_class: SimpleViT_LSA_ConvTok
  attention_type: lsa_diagonal
  arch_hash: 95ace50634b621d7
  image_size: 384
  channels: 1


In [6]:
copy_file_with_retries(ZIP_PATH, LOCAL_ZIP_PATH)

print('\nTesting local ZIP integrity...')
with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
    bad = zf.testzip()
    assert bad is None, f'Corrupted ZIP entry: {bad}'
print('Local ZIP verified.')

if LOCAL_DATA_ROOT.exists():
    print('\nRemoving old extracted dataset:', LOCAL_DATA_ROOT)
    shutil.rmtree(LOCAL_DATA_ROOT)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

print('\nExtracting local ZIP using system unzip...')
cmd = ['unzip', '-q', str(LOCAL_ZIP_PATH), '-d', str(LOCAL_DATA_ROOT)]
start = time.time()
result = subprocess.run(cmd, text=True, capture_output=True)
elapsed = time.time() - start
if result.returncode != 0:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)
    raise RuntimeError('unzip failed')

print(f'Extraction complete in {elapsed / 60:.2f} minutes.')
print('\nTop-level extracted contents:')
for p in sorted(LOCAL_DATA_ROOT.iterdir()):
    print(' ', p)


Copying preprocessed_384.zip to local disk, attempt 1/5...
  source size: 1709.02 MB
  local size:  1709.02 MB

Testing local ZIP integrity...
Local ZIP verified.

Extracting local ZIP using system unzip...
Extraction complete in 0.50 minutes.

Top-level extracted contents:
  /content/lsa_convtok_xai_run/data/preprocessed


In [7]:

def has_class_folder_structure(root: Path) -> bool:
    # True if root has at least two immediate class subfolders containing .npy files.
    if not root.exists() or not root.is_dir():
        return False
    class_dirs = [p for p in root.iterdir() if p.is_dir()]
    dirs_with_npy = []
    for d in class_dirs:
        try:
            if any(d.glob('*.npy')):
                dirs_with_npy.append(d)
        except Exception:
            pass
    return len(dirs_with_npy) >= 2

candidate_dataset_roots = []
if has_class_folder_structure(LOCAL_DATA_ROOT):
    candidate_dataset_roots.append(LOCAL_DATA_ROOT)
for p in LOCAL_DATA_ROOT.rglob('*'):
    if p.is_dir() and has_class_folder_structure(p):
        candidate_dataset_roots.append(p)

# Remove duplicates while preserving order.
seen = set()
unique_candidates = []
for p in candidate_dataset_roots:
    rp = p.resolve()
    if rp not in seen:
        unique_candidates.append(p)
        seen.add(rp)

print('Candidate dataset roots:')
for i, p in enumerate(unique_candidates):
    class_dirs = [d.name for d in p.iterdir() if d.is_dir() and any(d.glob('*.npy'))]
    n_npy = sum(1 for _ in p.rglob('*.npy'))
    print(f'[{i}] {p}')
    print('    classes:', class_dirs)
    print('    npy files:', n_npy)

assert unique_candidates, 'No dataset root found after extraction.'

# Use the first detected dataset root by default.
DATASET_ROOT = unique_candidates[0]
PREPROCESSED_ROOT_FOR_XAI = DATASET_ROOT.parent
DATASET_SUBDIR_FOR_XAI = DATASET_ROOT.name

print('\nUsing dataset root:', DATASET_ROOT)
print('Use --preprocessed-root:', PREPROCESSED_ROOT_FOR_XAI)
print('Use --dataset-subdir:', DATASET_SUBDIR_FOR_XAI)


Candidate dataset roots:
[0] /content/lsa_convtok_xai_run/data/preprocessed
    classes: ['Candida albicans', 'Candida glabrata']
    npy files: 3763

Using dataset root: /content/lsa_convtok_xai_run/data/preprocessed
Use --preprocessed-root: /content/lsa_convtok_xai_run/data
Use --dataset-subdir: preprocessed


In [8]:
split_candidates = []

# Preferred Drive locations first.
split_candidates.append(DRIVE_PREPROCESSED_ROOT / 'split_indices.json')
split_candidates.append(XAI_ROOT / 'split_indices.json')

# Local/extracted fallbacks.
split_candidates.append(LOCAL_DATA_ROOT / 'split_indices.json')
split_candidates += list(DRIVE_PREPROCESSED_ROOT.rglob('split_indices.json'))
split_candidates += list(LOCAL_DATA_ROOT.rglob('split_indices.json'))

SPLIT_FILE = None
for p in split_candidates:
    if p.exists():
        SPLIT_FILE = p
        break

assert SPLIT_FILE is not None, (
    'split_indices.json not found. Put it under xai/preprocessed/split_indices.json '
    'or inside the extracted dataset zip.'
)

with open(SPLIT_FILE, 'r') as f:
    split_data = json.load(f)

print('Split file:', SPLIT_FILE)
print('Split keys:', sorted(split_data.keys()))
assert 'test_base_paths' in split_data, 'split_indices.json must contain test_base_paths for --split test.'
print('Number of test entries:', len(split_data['test_base_paths']))


Split file: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/preprocessed/split_indices.json
Split keys: ['class_distribution', 'classes', 'constraints', 'drift_metrics', 'random_seed', 'sampling_method', 'test_base_paths', 'test_count', 'test_size', 'total_common_samples', 'train_base_paths', 'train_count']
Number of test entries: 821


In [9]:

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Runtime will be very slow. In Colab, select Runtime > Change runtime type > GPU.')


CUDA available: True
GPU: Tesla T4


In [10]:
BATCH_SIZE = 8
FAITHFULNESS_STEPS = 20
REVIEW_N_TP = 9
REVIEW_N_TN = 9
REVIEW_N_FP = 6
REVIEW_N_FN = -1  # include all available false negatives

LOCAL_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

FULL_CMD = [
    'python', str(LOCAL_SCRIPTS_DIR / 'analysis_attention.py'),
    '--project-root', str(LOCAL_SCRIPTS_DIR),
    '--preprocessed-root', str(PREPROCESSED_ROOT_FOR_XAI),
    '--dataset-subdir', str(DATASET_SUBDIR_FOR_XAI),
    '--output-root', str(LOCAL_OUTPUT_ROOT),
    '--split-file', str(SPLIT_FILE),
    '--experiment', EXPERIMENT,
    '--split', 'test',
    '--batch-size', str(BATCH_SIZE),
    '--run-embedding-analysis',
    '--compute-attention-rollout',
    '--compute-faithfulness',
    '--faithfulness-steps', str(FAITHFULNESS_STEPS),
    '--faithfulness-operators', 'zero', 'mean_patch',
    '--faithfulness-modes', 'insertion', 'deletion',
    '--faithfulness-target', 'predicted',
    '--faithfulness-ranking-source', 'rollout',
    '--faithfulness-controls', 'random', 'inverse',
    '--faithfulness-random-seed', '42',
    '--generate-review-panels',
    '--review-n-tp', str(REVIEW_N_TP),
    '--review-n-tn', str(REVIEW_N_TN),
    '--review-n-fp', str(REVIEW_N_FP),
    '--review-n-fn', str(REVIEW_N_FN),
    '--review-mmd-prototype-fraction', '0.6',
    '--review-embedding-dims', '8',
    '--review-random-seed', '42',
    '--review-slide-aware',
    '--resume-xai',
]

print('Full command:')
print(' '.join(FULL_CMD))
print('\nModel path expected by analysis_attention.py:')
print(LOCAL_OUTPUT_ROOT / EXPERIMENT / f'final_model_{EXPERIMENT}.pth')
print('Exists:', (LOCAL_OUTPUT_ROOT / EXPERIMENT / f'final_model_{EXPERIMENT}.pth').exists())


Full command:
python /content/lsa_convtok_xai_run/scripts/analysis_attention.py --project-root /content/lsa_convtok_xai_run/scripts --preprocessed-root /content/lsa_convtok_xai_run/data --dataset-subdir preprocessed --output-root /content/lsa_convtok_xai_run/output --split-file /content/drive/MyDrive/colab/lsa_convtok_vit/xai/preprocessed/split_indices.json --experiment lsa_convtok --split test --batch-size 8 --run-embedding-analysis --compute-attention-rollout --compute-faithfulness --faithfulness-steps 20 --faithfulness-operators zero mean_patch --faithfulness-modes insertion deletion --faithfulness-target predicted --faithfulness-ranking-source rollout --faithfulness-controls random inverse --faithfulness-random-seed 42 --generate-review-panels --review-n-tp 9 --review-n-tn 9 --review-n-fp 6 --review-n-fn -1 --review-mmd-prototype-fraction 0.6 --review-embedding-dims 8 --review-random-seed 42 --review-slide-aware --resume-xai

Model path expected by analysis_attention.py:
/content/l

In [11]:
process = subprocess.Popen(
    FULL_CMD,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

return_code = process.wait()

print('\n--- XAI process finished ---')
print('Return code:', return_code)
assert return_code == 0, 'XAI pipeline failed. Check the logs above.'


2026-05-08 10:57:50.846208: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Device Information:
  Device: cuda
  GPU: Tesla T4
  Memory: 15.64 GB
  CUDA Version: 12.8
NPYImageFolder initialized: 3763 samples (grayscale)
Dataset: 3763 samples | classes=['Candida albicans', 'Candida glabrata']
[LSA-ConvTok] Analytical output grid: (24, 24) (N=576) for 384×384 input
[LSA-ConvTok] Analytical matched-N passed: (24, 24) == (24, 24), N=576
Loaded model: SimpleViT_LSA_ConvTok
DataLoader: 821 samples | batch_size=8
Architecture: SimpleViT_LSA_ConvTok
Temporary XAI checkpoint dir: /tmp/vit_attention_spatial_gadrtgb4
Registered inline LSA rollout hooks for layers: [0, 1, 2, 3, 4, 5, 6, 7]

Extracting: 100%|██████████| 103/103 [18:44<00:00, 10.92s/it]
/usr/

In [12]:
print('Local analysis directory:', LOCAL_ANALYSIS_DIR)

expected_outputs = [
    LOCAL_ANALYSIS_DIR / 'embeddings_test_lsa_convtok.npz',
    LOCAL_ANALYSIS_DIR / 'rollout_stats_test.json',
    LOCAL_ANALYSIS_DIR / 'faithfulness_rollout_stats_test.json',
    LOCAL_ANALYSIS_DIR / 'faithfulness_rollout_per_sample_auc_test.csv',
    LOCAL_ANALYSIS_DIR / 'xai_summary_test.json',
]

for p in expected_outputs:
    print(('FOUND   ' if p.exists() else 'MISSING '), p)

print('Local analysis directory contents:')
for p in sorted(LOCAL_ANALYSIS_DIR.iterdir()):
    print(' ', p.name)


Local analysis directory: /content/lsa_convtok_xai_run/output/lsa_convtok/analysis
FOUND    /content/lsa_convtok_xai_run/output/lsa_convtok/analysis/embeddings_test_lsa_convtok.npz
FOUND    /content/lsa_convtok_xai_run/output/lsa_convtok/analysis/rollout_stats_test.json
FOUND    /content/lsa_convtok_xai_run/output/lsa_convtok/analysis/faithfulness_rollout_stats_test.json
FOUND    /content/lsa_convtok_xai_run/output/lsa_convtok/analysis/faithfulness_rollout_per_sample_auc_test.csv
FOUND    /content/lsa_convtok_xai_run/output/lsa_convtok/analysis/xai_summary_test.json
Local analysis directory contents:
  embeddings_test_lsa_convtok.npz
  embeddings_test_lsa_convtok.statistics.json
  faithfulness_auc_comparison_test.png
  faithfulness_inverse_mean_patch_deletion_test.png
  faithfulness_inverse_mean_patch_insertion_test.png
  faithfulness_inverse_zero_deletion_test.png
  faithfulness_inverse_zero_insertion_test.png
  faithfulness_random_mean_patch_deletion_test.png
  faithfulness_random_me

In [13]:
DRIVE_ANALYSIS_DIR = DRIVE_EXPERIMENT_DIR / 'analysis'

assert LOCAL_ANALYSIS_DIR.exists(), f'Local analysis dir missing: {LOCAL_ANALYSIS_DIR}'
DRIVE_ANALYSIS_DIR.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_ANALYSIS_DIR.exists():
    backup_dir = DRIVE_EXPERIMENT_DIR / ('analysis_backup_' + time.strftime('%Y%m%d_%H%M%S'))
    print('Existing Drive analysis folder found. Moving it to backup:')
    print(backup_dir)
    shutil.move(str(DRIVE_ANALYSIS_DIR), str(backup_dir))

print('Copying local analysis results back to Drive...')
shutil.copytree(LOCAL_ANALYSIS_DIR, DRIVE_ANALYSIS_DIR)
print('Copied results to:', DRIVE_ANALYSIS_DIR)

print('\nDrive analysis contents:')
for p in sorted(DRIVE_ANALYSIS_DIR.iterdir()):
    print(' ', p.name)


Copying local analysis results back to Drive...
Copied results to: /content/drive/MyDrive/colab/lsa_convtok_vit/xai/output/lsa_convtok/analysis

Drive analysis contents:
  embeddings_test_lsa_convtok.npz
  embeddings_test_lsa_convtok.statistics.json
  faithfulness_auc_comparison_test.png
  faithfulness_inverse_mean_patch_deletion_test.png
  faithfulness_inverse_mean_patch_insertion_test.png
  faithfulness_inverse_zero_deletion_test.png
  faithfulness_inverse_zero_insertion_test.png
  faithfulness_random_mean_patch_deletion_test.png
  faithfulness_random_mean_patch_insertion_test.png
  faithfulness_random_zero_deletion_test.png
  faithfulness_random_zero_insertion_test.png
  faithfulness_rollout_mean_patch_deletion_test.png
  faithfulness_rollout_mean_patch_insertion_test.png
  faithfulness_rollout_per_sample_auc_test.csv
  faithfulness_rollout_stats_test.json
  faithfulness_rollout_zero_deletion_test.png
  faithfulness_rollout_zero_insertion_test.png
  mmdcritic_FN_0481_rollout_test.pn

In [14]:

summary_path = DRIVE_ANALYSIS_DIR / 'xai_summary_test.json'
if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2)[:5000])
else:
    print('xai_summary_test.json not found yet.')


{
  "experiment": "lsa_convtok",
  "arch": "SimpleViT_LSA_ConvTok",
  "split": "test",
  "timestamp": "2026-05-08T11:47:14.067511",
  "stages_run": {
    "umap": true,
    "rollout": true,
    "faithfulness": true,
    "review_panels": true
  },
  "methodology": {
    "primary_explanation": "attention_rollout",
    "faithfulness_ranking_source": "attention_rollout",
    "faithfulness_score": "target_class_logit",
    "references": [
      "Abnar & Zuidema (2020) attention rollout/flow",
      "Samek et al. (2017) perturbation-based explanation evaluation",
      "Petsiuk et al. (2018) insertion/deletion AUC / RISE"
    ]
  },
  "umap": {
    "method": "UMAP",
    "trustworthiness_k5": 0.985067261147642,
    "silhouette_ground_truth": 0.626784086227417,
    "silhouette_predictions": 0.6765100955963135,
    "classification_accuracy": 0.9549330085261876,
    "slide_aware_metrics": {
      "n_slides": 12,
      "intra_slide_variance": 1.657354712486267,
      "inter_slide_separation_mean":